# Module 2: Principal Component Analysis

This notebook uses PCA to summarize 12 correlated measures of parent–young-adult support, connection, preparation, and independence. PCA converts the standardized variables into orthogonal principal components ordered by the variance they explain.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", palette="Set2")
data = pd.read_csv("module2_unlabeled_numeric_data.csv")
X = StandardScaler().fit_transform(data)
print(f"PCA input shape: {data.shape}")
data.head()

## Fit PCA

Standardization is necessary because PCA is variance-based. It prevents a variable with a larger numerical spread from controlling the components simply because of its scale. The eigenvectors become the component directions, and their eigenvalues determine the amount of variance explained.

In [ ]:
model = PCA()
scores = model.fit_transform(X)
variance = pd.DataFrame({
    "Component": np.arange(1, len(data.columns) + 1),
    "Explained variance": model.explained_variance_ratio_,
    "Cumulative variance": np.cumsum(model.explained_variance_ratio_)
})
variance.round(4)

## Visualization 1: scree and cumulative variance plot

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.bar(variance["Component"], variance["Explained variance"] * 100, color="#6F9F8C", alpha=0.85)
ax1.set(xlabel="Principal component", ylabel="Variance explained (%)", title="PCA Scree Plot")
ax1.set_xticks(variance["Component"])
ax2 = ax1.twinx()
ax2.plot(variance["Component"], variance["Cumulative variance"] * 100, marker="o", color="#E58D72", linewidth=2.5)
ax2.set_ylabel("Cumulative variance (%)")
ax2.axhline(80, color="#6A7E77", linestyle="--", alpha=0.7)
plt.show()
print(f"PC1 + PC2 explain {variance.loc[1, 'Cumulative variance']:.1%} of the standardized variance.")

## Component loadings

In [ ]:
loadings = pd.DataFrame(
    model.components_.T,
    index=data.columns,
    columns=[f"PC{i}" for i in range(1, len(data.columns) + 1)]
)
loadings[["PC1", "PC2", "PC3"]].round(3).sort_values("PC1", key=lambda s: s.abs(), ascending=False)

## Visualization 2: PCA biplot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(scores[:, 0], scores[:, 1], s=24, alpha=0.28, color="#6F9F8C")
scale = 4.0
for feature in data.columns:
    x, y = loadings.loc[feature, ["PC1", "PC2"]] * scale
    ax.arrow(0, 0, x, y, color="#2A7F78", alpha=0.75, head_width=0.08, length_includes_head=True)
    ax.text(x * 1.08, y * 1.08, feature, fontsize=8, color="#183F3A")
ax.axhline(0, color="#CFDDD7", linewidth=1)
ax.axvline(0, color="#CFDDD7", linewidth=1)
ax.set(
    title="PCA Biplot: Respondents and Variable Directions",
    xlabel=f"PC1 ({model.explained_variance_ratio_[0]:.1%})",
    ylabel=f"PC2 ({model.explained_variance_ratio_[1]:.1%})"
)
plt.show()

## Visualization 3: loading heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(loadings[["PC1", "PC2", "PC3", "PC4"]], annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("How Each Variable Contributes to the First Four Components")
plt.show()

## Conclusion

The first principal component primarily combines parental advice, emotional reliance, relationship quality, and authenticity, so it represents a broad **family connection and support** dimension. The second component gives stronger positive weight to financial independence, authenticity, relationship quality, and knowing a parent beyond the parental role, while several advice measures point in the opposite direction. It therefore captures an **autonomy-with-connection** contrast. The first two components explain more than half of the standardized variation, making a two-dimensional view useful, but additional components are needed to preserve roughly 80% of the information. PCA simplifies the relationships among many survey measures without claiming that every young adult follows the same route to independence.